# Chapter 4: Collaborative Evaluation
How to measure whether two or more people label things the same way. **No LLM needed** in this notebook.

## Run in Google Colab or locally
Upload this notebook alone and run cells from top to bottom. Setup installs missing packages;
all required sample data and helper functions are included below.
For Gemini calls, add `GEMINI_API_KEY` in Colab's key-shaped **Secrets** panel and enable notebook access,
or enter your key at the hidden prompt. Local runs use the single repository-root `.env`.
Never paste a key into a code cell. API calls use your own quota; notebook 04 needs no API key.
Students: **File → Save a copy in Drive** for your own copy and session.
Generated files live in the current session; download any results you want to keep before it ends.


In [ ]:
# Run once per fresh Colab session. Local users can run uv sync at the repo root.
import importlib.util
import subprocess
import sys

packages = {"google.genai": "google-genai", "dotenv": "python-dotenv",
            "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "rank_bm25": "rank-bm25"}
def available(module):
    try:
        return importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        return False
missing = [package for module, package in packages.items() if not available(module)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


In [ ]:
"""Small helpers shared by all notebooks (Gemini calls + data loading)."""
import json
import os
import time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Colab uses its own Secrets; local runs find the root .env.
try:
    from google.colab import userdata
except ImportError:
    userdata = None
if userdata is None:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / ".env").is_file():
            load_dotenv(directory / ".env", override=False)
            break

MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
EMBED_MODEL = os.getenv("GEMINI_EMBED_MODEL", "gemini-embedding-001")

_client = None


def client():
    """Create the Gemini client once and reuse it."""
    global _client
    if _client is None:
        key = os.getenv("GEMINI_API_KEY")
        if not key and userdata is not None:
            try:
                key = userdata.get("GEMINI_API_KEY")
            except Exception:
                pass
        if not key:
            from getpass import getpass
            key = getpass("Enter your own Gemini API key (hidden): ").strip()
        if not key:
            raise RuntimeError("GEMINI_API_KEY is missing. Add it to Colab Secrets or the root .env.")
        _client = genai.Client(api_key=key)
    return _client



def _generate(contents, system=None, temperature=0.0, as_json=False, model=None, retries=5):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json" if as_json else None,
    )
    for attempt in range(retries):
        try:
            resp = client().models.generate_content(model=model or MODEL, contents=contents, config=config)
            return json.loads(resp.text) if as_json else resp.text
        except Exception as e:  # rate limits (free tier) or temporary errors
            if attempt == retries - 1:
                raise
            wait = 10 * (attempt + 1)
            print(f"[retry {attempt + 1}] {type(e).__name__}: waiting {wait}s")
            time.sleep(wait)


def ask(prompt, system=None, temperature=0.0, as_json=False, model=None):
    """One prompt in, one answer out. as_json=True returns a Python dict."""
    return _generate(prompt, system, temperature, as_json, model)


def chat(history, system=None, temperature=0.0, model=None):
    """history = [{"role": "user" | "assistant", "content": "..."}, ...]"""
    contents = [
        types.Content(role="model" if m["role"] == "assistant" else "user",
                      parts=[types.Part(text=m["content"])])
        for m in history
    ]
    return _generate(contents, system, temperature, False, model)


def embed(texts):
    """Turn a list of texts into a numpy matrix (one row per text)."""
    r = client().models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in r.embeddings])


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# All sample prompts and data are embedded in this notebook.
def load_data():
    return {'ch4': {'annotator_a': ['Pass', 'Pass', 'Pass', 'Fail', 'Pass', 'Pass', 'Fail', 'Pass', 'Pass', 'Fail', 'Pass', 'Pass', 'Pass', 'Fail', 'Pass', 'Pass', 'Fail', 'Pass', 'Pass', 'Pass'], 'annotator_b': ['Pass', 'Pass', 'Fail', 'Fail', 'Pass', 'Pass', 'Fail', 'Pass', 'Pass', 'Pass', 'Pass', 'Pass', 'Pass', 'Fail', 'Pass', 'Fail', 'Fail', 'Pass', 'Pass', 'Pass'], 'annotator_c': ['Pass', 'Pass', 'Pass', 'Fail', 'Pass', 'Fail', 'Fail', 'Pass', 'Pass', 'Fail', 'Pass', 'Pass', 'Pass', 'Fail', 'Pass', 'Pass', 'Fail', 'Pass', 'Fail', 'Pass'], 'scores_a': [5, 4, 4, 2, 5, 3, 1, 4, 5, 2, 4, 4, 5, 1, 4, 3, 2, 5, 4, 5], 'scores_b': [5, 5, 3, 1, 4, 3, 2, 4, 5, 3, 4, 5, 4, 2, 4, 2, 1, 5, 4, 5]}}


In [ ]:
from collections import Counter
import numpy as np

data = load_data()["ch4"]
a, b, c3 = data["annotator_a"], data["annotator_b"], data["annotator_c"]
print("Annotator A:", a)
print("Annotator B:", b)

## 1. Percent agreement (simple but misleading)

In [ ]:
def percent_agreement(r1, r2):
    return sum(x == y for x, y in zip(r1, r2)) / len(r1)

print(f"A vs B percent agreement: {percent_agreement(a, b):.0%}")

## 2. Cohen's kappa (2 annotators, removes luck)
`kappa = (Po - Pe) / (1 - Pe)`
- `Po` = observed agreement
- `Pe` = agreement expected by pure chance

In [ ]:
def cohens_kappa(r1, r2):
    n = len(r1)
    po = sum(x == y for x, y in zip(r1, r2)) / n              # observed
    c1, c2 = Counter(r1), Counter(r2)
    pe = sum((c1[l] / n) * (c2[l] / n) for l in set(r1) | set(r2))   # by chance
    return (po - pe) / (1 - pe)

print(f"A vs B kappa: {cohens_kappa(a, b):.2f}")

### Check with the book example
100 summaries: both Pass = 75, A Pass/B Fail = 10, A Fail/B Pass = 5, both Fail = 10. Book answer: **kappa ≈ 0.48**.

In [ ]:
r1 = ["Pass"] * 75 + ["Pass"] * 10 + ["Fail"] * 5 + ["Fail"] * 10
r2 = ["Pass"] * 75 + ["Fail"] * 10 + ["Pass"] * 5 + ["Fail"] * 10

print(f"Percent agreement: {percent_agreement(r1, r2):.2f}")
print(f"Kappa            : {cohens_kappa(r1, r2):.2f}")

## 3. Why percent agreement can lie
95 of 100 traces are Pass. The two annotators disagree on **every** Fail.

In [ ]:
x = ["Pass"] * 95 + ["Fail"] * 5          # annotator X
y = ["Fail"] * 5 + ["Pass"] * 95          # annotator Y (fails different traces)

print(f"Percent agreement: {percent_agreement(x, y):.0%}   <- looks great")
print(f"Kappa            : {cohens_kappa(x, y):.2f}   <- reality: no real agreement")

## 4. Compare with scikit-learn (should match)

In [ ]:
from sklearn.metrics import cohen_kappa_score
print("Ours   :", round(cohens_kappa(a, b), 3))
print("sklearn:", round(cohen_kappa_score(a, b), 3))

## 5. Weighted kappa (ordered scores 1-5)
Being 1 point off is better than 4 points off, so we use weights.

In [ ]:
sa, sb = data["scores_a"], data["scores_b"]
print("Plain kappa    :", round(cohen_kappa_score(sa, sb), 2))
print("Weighted kappa :", round(cohen_kappa_score(sa, sb, weights="quadratic"), 2))

## 6. Fleiss' kappa (3 or more annotators)

In [ ]:
def fleiss_kappa(ratings):
    """ratings = list of items; each item = list of labels from all annotators"""
    labels = sorted({x for item in ratings for x in item})
    n_raters, n_items = len(ratings[0]), len(ratings)
    counts = np.array([[item.count(l) for l in labels] for item in ratings])
    p_label = counts.sum(axis=0) / (n_items * n_raters)
    p_item = ((counts ** 2).sum(axis=1) - n_raters) / (n_raters * (n_raters - 1))
    pe = (p_label ** 2).sum()
    return (p_item.mean() - pe) / (1 - pe)

items = list(zip(a, b, c3))                  # each item = (A label, B label, C label)
items = [list(i) for i in items]
print(f"Fleiss kappa (A, B, C): {fleiss_kappa(items):.2f}")

## 7. Read the number + find disagreements for the alignment session

In [ ]:
def interpret(k):
    if k < 0: return "poor"
    if k <= 0.20: return "slight"
    if k <= 0.40: return "fair"
    if k <= 0.60: return "moderate"
    if k <= 0.80: return "substantial"
    return "almost perfect"

k = cohens_kappa(a, b)
print(f"kappa = {k:.2f} -> {interpret(k)}  (aim for 0.6 or more)")

print("\nTraces to discuss (A and B disagree):")
for i, (x, y) in enumerate(zip(a, b), start=1):
    if x != y:
        print(f"  trace {i}: A={x}, B={y}")